In [ ]:
from PIL import Image //The Python Imaging Library (PIL) is a popular library for image processing in Python.
import requests

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils[decord]

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# 1. Load the model - we use 4-bit quantization to make sure it fits in the free RAM
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa", # Makes it faster
    device_map="auto",
)

# 2. Load the processor (the 'translator' between image and text)
processor = AutoProcessor.from_pretrained(model_id)

# 3. Define your prompt and image
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://media.licdn.com/dms/image/v2/D4E12AQENIiXVh9AW0Q/article-cover_image-shrink_720_1280/B4EZhZqBRvGcAM-/0/1753850844868?e=2147483647&v=beta&t=L1Tw-PvxGj2BMp1gFplbVUkK2v8LY6z40UgPqkXV_To", # Feel free to change this!
            },
            {"type": "text", "text": "Describe what is happening in this image."},
        ],
    }
]

# 4. Prepare the inputs
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to("cuda")

# 5. Generate the answer
generated_ids = model.generate(**inputs, max_new_tokens=512)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print("-" * 30)
print(output_text[0])

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

------------------------------
The image appears to be a table comparing three different types of models: LLM (Large Language Model), VLM (Vision-Language Model), and LAMM (Language-Action Model). Here's a breakdown of the table:

### Columns:
1. **Input**: The type of input data used by each model.
2. **Output**: The type of output generated by each model.
3. **Objective**: The primary task or goal that each model aims to achieve.
4. **Boundary**: The domain or area where each model operates.

### Rows:
1. **LLM**:
   - **Input**: Text
   - **Output**: Text
   - **Objective**: Next-token prediction
   - **Boundary**: Conversational

2. **VLM**:
   - **Input**: Text + Images/Video
   - **Output**: Text + Images/Video
   - **Objective**: Vision-language alignment
   - **Boundary**: Perception

3. **LAMM**:
   - **Input**: Text + Images/Video
   - **Output**: Concrete actions (e.g., keystrokes, API calls)
   - **Objective**: Policy learning (imitation + reinforcement learning)
   - **Bou

### **Now working with videos:**

In [ ]:
!pip install -q av

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# 1. Setup Model (Same as before, using 'sdpa' for Colab compatibility)
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

# 2. Define Video Input
video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle-car-detection.mp4"
# You can use a URL or a local path like "file:///content/my_video.mp4"
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": video_url,
                "fps": 1.0, # Sample 1 frame per second of video
                "max_pixels": 360 * 420, # Keep resolution low for T4 GPU
            },
            {"type": "text", "text": "Describe the events in this video step by step."},
        ],
    }
]

# 3. Process & Generate
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to("cuda")

generated_ids = model.generate(**inputs, max_new_tokens=256)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print("\n--- Video Analysis ---\n")
print(output_text[0])

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]


--- Video Analysis ---

A man walks across a parking lot. He stops and looks around. A white car drives into the parking lot. The man continues walking. A person on a bicycle rides into the parking lot. The man continues walking. A blue car drives into the parking lot. The man continues walking.
